# Feature Extraction

This notebook creates the video-level feature dataset for **valence and arousal analysis/prediction**.

The final dataset follows this structure:

- one row = one participant × one empathy scene;
- targets = `valence1–4` and `arousal1–4` from `final_data_questionnaires.csv`;
- sensor features = only `min` and `max`;
- scene extraction = fallback split into 6 approximate segments;
- only empathy scenes 1–4 are used for the supervised dataset.

The split-6 approach follows the practical guidance from the project communication: after the calibration part is excluded by using the video start time, the video session is divided into six parts: forest baseline, four empathy scenes, and roller coaster. To reduce transition/questionnaire noise, approximately 40 seconds are trimmed from segment boundaries.

This corrected version also includes a fallback for participants with missing `end_vid_time`: if the end time is missing, the last available timestamp from the participant sensor files is used as the approximate session end.


In [1]:
from pathlib import Path
import re
import gc
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [3]:
from pathlib import Path


CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = None

for candidate in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if (candidate / "data" / "data3").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/data3. "
    )

DATA_DIR = PROJECT_ROOT/"data"/"data3"

QUESTIONNAIRES_PATH = DATA_DIR / "final_data_questionnaires.csv"
DETAILS_PATH = DATA_DIR / "final_data_details.csv"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

FINAL_OUTPUT_PATH = OUTPUT_DIR / "video_level_valence_arousal_minmax.csv"
LOG_OUTPUT_PATH = OUTPUT_DIR / "feature_extraction_logs.csv"

print("Current dir:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Questionnaires exists:", QUESTIONNAIRES_PATH.exists())
print("Details exists:", DETAILS_PATH.exists())
print("Output file:", FINAL_OUTPUT_PATH)
print("Log file:", LOG_OUTPUT_PATH)

Current dir: /Users/victoriaprojkova/EmteqPRO-VR
Project root: /Users/victoriaprojkova/EmteqPRO-VR
Data dir: /Users/victoriaprojkova/EmteqPRO-VR/data/data3
Questionnaires exists: True
Details exists: True
Output file: /Users/victoriaprojkova/EmteqPRO-VR/outputs/video_level_valence_arousal_minmax.csv
Log file: /Users/victoriaprojkova/EmteqPRO-VR/outputs/feature_extraction_logs.csv


## Feature extraction configuration

This section defines the main extraction strategy.

The notebook uses the approved fallback approach:

1. start from the video session start time;
2. estimate the video session end time;
3. divide the interval into 6 segments;
4. trim each segment to reduce transition/questionnaire noise;
5. keep only empathy scenes 1–4;
6. extract only `min` and `max` features.


In [14]:
N_SCENES = 6
TRIM_SECONDS = 40 #for the questionnaire data

MIN_WINDOW_SECONDS = 30 #it the predefined trimming seconds cut out much relevant info, automatically change to 30 seconds 

# First test with 3 participants. Then set to None for all participants.
MAX_PARTICIPANTS = None 

SCENE_ORDER = [
    "forest_baseline",
    "empathy_scene_1",
    "empathy_scene_2",
    "empathy_scene_3",
    "empathy_scene_4",
    "roller_coaster",
]

EMPATHY_SCENES = [
    "empathy_scene_1",
    "empathy_scene_2",
    "empathy_scene_3",
    "empathy_scene_4",
]

SCENE_TO_LABEL_INDEX = {
    "empathy_scene_1": 1,
    "empathy_scene_2": 2,
    "empathy_scene_3": 3,
    "empathy_scene_4": 4,
}

FEATURE_FILE_PREFIXES = [
    "emgActivation",
    "facialActivation",
    "expression",
    "breathingRate",
    "hrv",
]

# It will only be used if it contains a Time column.
OPTIONAL_FEATURE_FILE_PREFIXES = [
    "headMotion",
]

#excluded since we would like to use the arousal and valence as target labels (the ones from the questionnaires) - if included, data leakage would happen
EXCLUDED_PREFIXES = [
    "arousal",
    "valence",
    "facialValence",
]


## Load questionnaire labels and timing metadata

This section loads the two main metadata files.

`final_data_questionnaires.csv` contains the target labels:

- `valence1`, `valence2`, `valence3`, `valence4`;
- `arousal1`, `arousal2`, `arousal3`, `arousal4`.

`final_data_details.csv` contains timing information:

- calibration start time;
- video start time;
- video end time when available.

If the video end time is missing for some participants, a fallback based on the last available sensor timestamp is applied later.


In [5]:
questionnaires_df = pd.read_csv(QUESTIONNAIRES_PATH)
details_df = pd.read_csv(DETAILS_PATH)

questionnaires_df["ID"] = pd.to_numeric(questionnaires_df["ID"], errors="coerce").astype("Int64")
details_df["ID"] = pd.to_numeric(details_df["ID"], errors="coerce").astype("Int64")

print("Questionnaires shape:", questionnaires_df.shape)
print("Details shape:", details_df.shape)

display(questionnaires_df.head())
display(details_df.head())


Questionnaires shape: (105, 68)
Details shape: (105, 8)


,ID,gender,age,slo_fluent,epil-hc,other_healthcon,good_vision,bad_hearing,facial_palsy-stroke,VR_experience,current_status,degree,group_job-sp,anxiety,inf_cons,video,feelof_presence_inVE,captivatedby_VE,real_VWorld,awareness_RWorld,issues_in_VR,stateEmpathy1a,stateEmpathy1b,stateEmpathy1c,valence1,arousal1,discomfort1,stateEmpathy2a,stateEmpathy2b,stateEmpathy2c,valence2,arousal2,discomfort2,stateEmpathy3a,stateEmpathy3b,stateEmpathy3c,valence3,arousal3,discomfort3,stateEmpathy4a,stateEmpathy4b,stateEmpathy4c,valence4,arousal4,discomfort4,stateValence,stateArousal,perspective-taking_points,perspective-taking_rate,online-simulation_points,online-simulation_rate,traitEmpathy_cog_points,traitEmpathy_cog_rate,emotion-contagion_points,emotion-contagion_rate,proximal-responsivity_points,proximal-responsivity_rate,peripheral-responsivity_points,peripheral-responsivity_rate,traitEmpathy_affec_points,traitEmpathy_affec_rate,Empathy1,Empathy2,Empathy3,Empathy4,Empathy_avg,Empathy_min,Arousal_RC
0,1,z,24,1,0,0,1,0,0,3,e,4,14,0,1,3,8,8,7,3,4,3,3,3,4,3,3,2,1,2,3,2,3,3,2,3,0,2,0,2,2,3,4,3,3,0.0,4.0,14,1.4,14,1.56,2.96,1,11,2.75,7,1.75,14,3.50,8.0,3,3.0,1.7,2.7,2.3,2.4,1.7,4.0
1,2,m,27,1,0,0,1,0,0,3,s,4,14,1,1,1,7,7,6,3,7,3,3,3,3,3,3,0,0,0,2,2,0,1,1,1,1,1,0,2,1,2,3,1,1,2.0,0.0,27,2.7,21,2.33,5.03,3,9,2.25,9,2.25,10,2.50,7.0,2,3.0,0.0,1.0,1.7,1.4,0.0,0.0
2,3,m,28,1,0,0,1,0,0,3,e,5,10,0,1,3,3,3,2,2,7,1,1,1,1,3,1,3,3,2,3,3,2,3,3,2,0,0,0,1,2,2,2,1,1,NaN,NaN,21,2.1,16,1.78,3.88,2,10,2.50,11,2.75,11,2.75,8.0,3,1.0,2.7,2.7,1.7,2.0,1.0,NaN
3,4,m,30,1,0,0,1,0,0,1,e,4,14,0,1,1,3,3,3,5,1,2,2,3,4,2,2,3,3,3,3,3,3,3,2,4,1,2,0,2,3,3,3,2,2,2.0,1.0,17,1.7,25,2.78,4.48,2,9,2.25,10,2.50,11,2.75,7.5,2,2.3,3.0,3.0,2.7,2.8,2.3,1.0
4,5,m,35,1,0,0,1,0,0,2,e,5,14,0,1,5,8,4,8,2,7,3,1,3,3,3,4,2,2,4,3,3,3,3,4,4,2,0,0,0,0,4,4,3,3,0.0,4.0,16,1.6,17,1.89,3.49,2,6,1.50,6,1.50,10,2.50,5.5,2,2.3,2.7,3.7,1.3,2.5,1.3,4.0


,ID,start_cal_time_h,start_cal_time_m,start_vid_time_h,start_vid_time_m,end_vid_time_h,end_vid_time_m,additional info
0,1,14,1,14,5,14.0,36.0,work
1,2,13,6,13,10,13.0,38.0,work
2,3,14,11,14,15,14.0,35.0,work - without RC vid
3,4,13,9,13,13,13.0,34.0,work
4,5,14,18,14,22,14.0,47.0,work


In [6]:
required_questionnaire_cols = ["ID", "video"]
for i in range(1, 5):
    required_questionnaire_cols += [f"valence{i}", f"arousal{i}"]

missing_q = [c for c in required_questionnaire_cols if c not in questionnaires_df.columns]
if missing_q:
    raise ValueError(f"Missing questionnaire columns: {missing_q}")

required_details_cols = [
    "ID",
    "start_vid_time_h",
    "start_vid_time_m",
]

missing_d = [c for c in required_details_cols if c not in details_df.columns]
if missing_d:
    raise ValueError(f"Missing details columns: {missing_d}")

print("Required questionnaire and timing columns are available.")


Required questionnaire and timing columns are available.


## Helper functions

This section defines small utility functions used throughout the notebook.

These functions handle:

- extracting participant IDs from folder names;
- cleaning feature names;
- reading CSV files robustly;
- converting hour/minute values into minutes.


In [7]:
def extract_participant_id_from_folder(folder_name: str):
    """Extract numeric participant ID from folder names such as participant1 or participant12-2."""
    match = re.search(r"participant(\d+)", folder_name, flags=re.IGNORECASE)
    if not match:
        return None
    return int(match.group(1))


def extract_recording_suffix_from_folder(folder_name: str):
    """Extract optional suffix from folder names such as participant12-2."""
    match = re.search(r"participant\d+-(\d+)", folder_name, flags=re.IGNORECASE)
    if not match:
        return 0
    return int(match.group(1))


def clean_column_name(name: str) -> str:
    """Convert a column name into a safe feature name."""
    name = str(name).strip()
    name = name.replace("#", "number")
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name.lower()


def read_csv_smart(path: Path) -> pd.DataFrame:
    """Read CSV files robustly."""
    try:
        df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path, sep=",")

    if df.shape[1] == 1:
        try:
            df = pd.read_csv(path, sep=",")
        except Exception:
            pass

    return df


def time_hm_to_minutes(hour, minute):
    """Convert hour/minute to minutes from midnight."""
    if pd.isna(hour) or pd.isna(minute):
        return np.nan
    return int(hour) * 60 + int(minute)


## Sensor timestamp helper functions

This section finds the first and last available sensor timestamps for each participant.

The first available timestamp is used to align the start of the video session with the sensor files.  
The last available timestamp is used as a fallback for participants where `end_vid_time` is missing.


In [8]:
def get_first_available_sensor_time(participant_folder: Path):
    """Return the first available Time value from any participant CSV with a Time column."""
    csv_files = sorted(participant_folder.glob("*.csv"))

    min_times = []
    for path in csv_files:
        try:
            df = read_csv_smart(path)
            if "Time" in df.columns and len(df) > 0:
                t = pd.to_numeric(df["Time"], errors="coerce").dropna()
                if len(t) > 0:
                    min_times.append(float(t.min()))
        except Exception:
            continue

    if len(min_times) == 0:
        return np.nan

    return min(min_times)


def get_last_available_sensor_time(participant_folder: Path):
    """Return the last available Time value from any participant CSV with a Time column."""
    csv_files = sorted(participant_folder.glob("*.csv"))

    max_times = []
    for path in csv_files:
        try:
            df = read_csv_smart(path)
            if "Time" in df.columns and len(df) > 0:
                t = pd.to_numeric(df["Time"], errors="coerce").dropna()
                if len(t) > 0:
                    max_times.append(float(t.max()))
        except Exception:
            continue

    if len(max_times) == 0:
        return np.nan

    return max(max_times)


## Estimate video start and end timestamps

This section estimates the video session start and end timestamps in the same time scale as the sensor files.

Preferred case:

- use `start_vid_time` and `end_vid_time` from `final_data_details.csv`.

Fallback case:

- if `end_vid_time` is missing, use the last available sensor timestamp as the approximate end of the recording.

This correction prevents participants with missing `end_vid_time` from being automatically removed (there were few participant with missing end_vid_time info).


In [9]:
def estimate_video_start_end_timestamps(participant_folder: Path, details_row: pd.Series):
    """
    Estimate video_start and video_end in the sensor timestamp scale.

    If end_vid_time is missing, use the last available sensor timestamp as fallback video_end.
    """
    first_sensor_time = get_first_available_sensor_time(participant_folder)
    last_sensor_time = get_last_available_sensor_time(participant_folder)

    if pd.isna(first_sensor_time):
        return np.nan, np.nan, "missing_first_sensor_time"

    if "start_cal_time_h" in details_row.index and "start_cal_time_m" in details_row.index:
        cal_minutes = time_hm_to_minutes(
            details_row["start_cal_time_h"],
            details_row["start_cal_time_m"]
        )
        vid_minutes = time_hm_to_minutes(
            details_row["start_vid_time_h"],
            details_row["start_vid_time_m"]
        )

        if pd.isna(cal_minutes) or pd.isna(vid_minutes):
            return np.nan, np.nan, "missing_start_time"

        offset_minutes = vid_minutes - cal_minutes
        if offset_minutes < 0:
            offset_minutes += 24 * 60

        video_start = first_sensor_time + offset_minutes * 60
    else:
        video_start = first_sensor_time

    start_vid_minutes = time_hm_to_minutes(
        details_row.get("start_vid_time_h", np.nan),
        details_row.get("start_vid_time_m", np.nan)
    )
    end_vid_minutes = time_hm_to_minutes(
        details_row.get("end_vid_time_h", np.nan),
        details_row.get("end_vid_time_m", np.nan)
    )

    if not pd.isna(start_vid_minutes) and not pd.isna(end_vid_minutes):
        duration_minutes = end_vid_minutes - start_vid_minutes
        if duration_minutes < 0:
            duration_minutes += 24 * 60

        video_end = video_start + duration_minutes * 60
        return float(video_start), float(video_end), "details_start_end"

    if not pd.isna(last_sensor_time) and last_sensor_time > video_start:
        video_end = last_sensor_time
        return float(video_start), float(video_end), "fallback_last_sensor_time"

    return float(video_start), np.nan, "missing_video_end"


## Build the split 6 scene schedule

This section divides the estimated video session interval into six approximate scene segments.

The six segments are:

1. forest baseline;
2. empathy scene 1;
3. empathy scene 2;
4. empathy scene 3;
5. empathy scene 4;
6. roller coaster.

Only empathy scenes 1–4 are used for the final labelled dataset.  
Each segment is trimmed by approximately 40 seconds from both sides to reduce transition and questionnaire noise. If a segment is too short, the trim is automatically reduced.


In [10]:
def build_split6_schedule(video_start: float, video_end: float):
    """Divide the video session into 6 approximate parts and trim each segment."""
    if pd.isna(video_start) or pd.isna(video_end) or video_end <= video_start:
        return pd.DataFrame()

    session_duration = video_end - video_start
    segment_duration = session_duration / N_SCENES

    rows = []
    for idx, scene_name in enumerate(SCENE_ORDER):
        seg_start = video_start + idx * segment_duration
        seg_end = video_start + (idx + 1) * segment_duration

        effective_trim = TRIM_SECONDS
        if segment_duration - 2 * effective_trim < MIN_WINDOW_SECONDS:
            effective_trim = max((segment_duration - MIN_WINDOW_SECONDS) / 2, 0)

        window_start = seg_start + effective_trim
        window_end = seg_end - effective_trim

        rows.append({
            "scene_name": scene_name,
            "scene_index_in_session": idx + 1,
            "segment_start": seg_start,
            "segment_end": seg_end,
            "window_start": window_start,
            "window_end": window_end,
            "segment_duration_seconds": segment_duration,
            "window_duration_seconds": window_end - window_start,
            "trim_seconds_used": effective_trim,
        })

    return pd.DataFrame(rows)


## Select feature files

This section determines which participant CSV files will be used for feature extraction.

Included in the first clean version:

- `emgActivation_*.csv`
- `facialActivation_*.csv`
- `expression_*.csv`
- `breathingRate_*.csv`
- `hrv_*.csv`

Excluded from the first model:

- `arousal_*.csv`
- `valence_*.csv`
- `facialValence_*.csv`


In [11]:
def get_file_type(path: Path):
    """Infer file type from filename."""
    name = path.name

    for prefix in EXCLUDED_PREFIXES:
        if name.startswith(prefix + "_") or name.endswith(f"_{prefix}.csv"):
            return prefix

    for prefix in FEATURE_FILE_PREFIXES:
        if name.startswith(prefix + "_") or name.endswith(f"_{prefix}.csv"):
            return prefix

    for prefix in OPTIONAL_FEATURE_FILE_PREFIXES:
        if name.startswith(prefix + "_") or name.endswith(f"_{prefix}.csv"):
            return prefix

    return None


def select_participant_files(participant_folder: Path):
    """Select files used for feature extraction and log skipped files."""
    selected = []
    logs = []

    for path in sorted(participant_folder.glob("*.csv")):
        file_type = get_file_type(path)

        if file_type in EXCLUDED_PREFIXES:
            logs.append({"file": path.name, "file_type": file_type, "status": "excluded_affect_output", "rows_in_window": 0})
            continue

        if file_type in FEATURE_FILE_PREFIXES:
            selected.append((path, file_type))
            continue

        if file_type in OPTIONAL_FEATURE_FILE_PREFIXES:
            try:
                df_head = read_csv_smart(path).head(5)
                if "Time" in df_head.columns:
                    selected.append((path, file_type))
                else:
                    logs.append({"file": path.name, "file_type": file_type, "status": "optional_no_time_column", "rows_in_window": 0})
            except Exception as e:
                logs.append({"file": path.name, "file_type": file_type, "status": "optional_read_failed", "error": str(e), "rows_in_window": 0})
            continue

        logs.append({"file": path.name, "file_type": file_type, "status": "unknown_file_type", "rows_in_window": 0})

    return selected, logs


## Feature extraction from one sensor file

This section defines how features are extracted from one CSV file inside a specific scene window.

For numeric files, the notebook extracts `min` and `max` from all usable numeric columns.  
For `expression_*.csv`, the notebook extracts global expression intensity and expression-specific intensity features.

`Ppg/QualityIndex` is kept only as quality-control metadata and is not treated as a main model feature.


In [12]:
def extract_numeric_minmax(df_window: pd.DataFrame, file_type: str):
    """Extract min/max from numeric columns in a window."""
    features = {}

    if df_window.empty:
        return features

    exclude_cols = {"Frame#", "Frame", "Time", "Ppg/QualityIndex"}
    numeric_df = df_window.copy()

    numeric_cols = []
    for col in numeric_df.columns:
        if col in exclude_cols:
            continue
        numeric_df[col] = pd.to_numeric(numeric_df[col], errors="coerce")
        if numeric_df[col].notna().sum() > 0:
            numeric_cols.append(col)

    prefix_clean = clean_column_name(file_type)

    for col in numeric_cols:
        values = numeric_df[col].dropna()
        if len(values) == 0:
            continue
        col_clean = clean_column_name(col)
        features[f"{prefix_clean}_{col_clean}_min"] = float(values.min())
        features[f"{prefix_clean}_{col_clean}_max"] = float(values.max())

    if "Ppg/QualityIndex" in df_window.columns:
        q = pd.to_numeric(df_window["Ppg/QualityIndex"], errors="coerce").dropna()
        if len(q) > 0:
            features[f"qc_{prefix_clean}_ppg_quality_min"] = float(q.min())
            features[f"qc_{prefix_clean}_ppg_quality_mean"] = float(q.mean())

    return features


def extract_expression_features(df_window: pd.DataFrame, file_type: str):
    """Extract expression intensity min/max globally and per Expression/Type."""
    features = {}

    if df_window.empty:
        return features

    type_col = "Expression/Type"
    intensity_col = "Expression/Intensity"
    prefix_clean = clean_column_name(file_type)

    if intensity_col not in df_window.columns:
        return features

    intensity = pd.to_numeric(df_window[intensity_col], errors="coerce")
    if intensity.notna().sum() > 0:
        features[f"{prefix_clean}_expression_intensity_min"] = float(intensity.min())
        features[f"{prefix_clean}_expression_intensity_max"] = float(intensity.max())

    if type_col in df_window.columns:
        tmp = df_window[[type_col, intensity_col]].copy()
        tmp[intensity_col] = pd.to_numeric(tmp[intensity_col], errors="coerce")
        tmp = tmp.dropna(subset=[type_col, intensity_col])

        for expr_type, group in tmp.groupby(type_col):
            expr_clean = clean_column_name(expr_type)
            vals = group[intensity_col].dropna()
            if len(vals) == 0:
                continue
            features[f"{prefix_clean}_{expr_clean}_intensity_min"] = float(vals.min())
            features[f"{prefix_clean}_{expr_clean}_intensity_max"] = float(vals.max())

    return features


def extract_features_from_file(path: Path, file_type: str, window_start: float, window_end: float):
    """Read one CSV and extract min/max features inside the selected time window."""
    df = read_csv_smart(path)

    if "Time" not in df.columns:
        return {}, {"file": path.name, "file_type": file_type, "status": "skipped_no_time", "rows_in_window": 0}

    df["Time"] = pd.to_numeric(df["Time"], errors="coerce")
    df = df.dropna(subset=["Time"])
    df_window = df[(df["Time"] >= window_start) & (df["Time"] < window_end)].copy()

    if df_window.empty:
        return {}, {"file": path.name, "file_type": file_type, "status": "empty_window", "rows_in_window": 0}

    if file_type == "expression":
        features = extract_expression_features(df_window, file_type)
    else:
        features = extract_numeric_minmax(df_window, file_type)

    return features, {
        "file": path.name,
        "file_type": file_type,
        "status": "ok",
        "rows_in_window": len(df_window),
        "time_min": float(df_window["Time"].min()),
        "time_max": float(df_window["Time"].max()),
    }


## Discover participant folders

This section finds all participant folders inside `data/data3`.

It also checks which files will be selected for feature extraction for each participant. If a participant has multiple folders, for example `participant12-1` and `participant12-2`, the notebook stores both the participant ID and the folder name so the situation can be inspected later.


In [13]:
participant_folders = [
    p for p in DATA_DIR.iterdir()
    if p.is_dir() and p.name.lower().startswith("participant")
]

participant_folders = sorted(
    participant_folders,
    key=lambda p: (
        extract_participant_id_from_folder(p.name) or 9999,
        extract_recording_suffix_from_folder(p.name),
        p.name
    )
)

participant_summary = []
for folder in participant_folders:
    pid = extract_participant_id_from_folder(folder.name)
    suffix = extract_recording_suffix_from_folder(folder.name)
    selected, skipped_logs = select_participant_files(folder)
    participant_summary.append({
        "participant_folder": folder.name,
        "participant_id": pid,
        "recording_suffix": suffix,
        "n_csv_files": len(list(folder.glob("*.csv"))),
        "n_selected_feature_files": len(selected),
        "selected_files": [p.name for p, _ in selected],
    })

participant_summary_df = pd.DataFrame(participant_summary)

print("Participant folders found:", len(participant_summary_df))
display(participant_summary_df.head(20))

duplicate_ids = participant_summary_df["participant_id"].value_counts()
duplicate_ids = duplicate_ids[duplicate_ids > 1]
if len(duplicate_ids) > 0:
    print("Participant IDs with multiple folders:")
    display(participant_summary_df[participant_summary_df["participant_id"].isin(duplicate_ids.index)])
else:
    print("No duplicate participant folder IDs detected.")


Participant folders found: 106


,participant_folder,participant_id,recording_suffix,n_csv_files,n_selected_feature_files,selected_files
0,participant1,1,0,10,5,"[breathingRate_3NVLIWA4.csv, emgActivation_3NV..."
1,participant2,2,0,10,5,"[breathingRate_S76VRM61.csv, emgActivation_S76..."
2,participant3,3,0,10,5,"[breathingRate_K37JKM9A.csv, emgActivation_K37..."
3,participant4,4,0,9,5,"[2024-01-09T13-09-17_breathingRate.csv, 2024-0..."
4,participant5,5,0,10,5,"[breathingRate_A7RGPTAZ.csv, emgActivation_A7R..."
5,participant6,6,0,10,5,"[breathingRate_IFGAU350.csv, emgActivation_IFG..."
6,participant7,7,0,9,5,"[2024-01-18T15-15-19_breathingRate.csv, 2024-0..."
7,participant8,8,0,9,5,"[2024-01-22T14-22-33_breathingRate.csv, 2024-0..."
8,participant9,9,0,9,5,"[2024-01-22T15-18-10_breathingRate.csv, 2024-0..."
9,participant10,10,0,10,5,"[breathingRate_PCGJ7G6L.csv, emgActivation_PCG..."


Participant IDs with multiple folders:


,participant_folder,participant_id,recording_suffix,n_csv_files,n_selected_feature_files,selected_files
11,participant12-1,12,1,10,5,"[breathingRate_HYJY9X99.csv, emgActivation_HYJ..."
12,participant12-2,12,2,8,5,"[2024-01-26T13-38-23_breathingRate.csv, 2024-0..."


## participant-level extraction

This section processes one participant folder at a time.

For each participant:

1. questionnaire targets are loaded;
2. video start/end timestamps are estimated;
3. the session is divided into 6 trimmed segments;
4. only empathy scenes 1–4 are used;
5. min/max features are extracted from the selected CSV files;
6. one row is created for each participant-empathy-scene pair.

The extraction logs are saved separately so that missing files, empty windows, and excluded files can be inspected.


In [15]:
def process_one_participant(participant_folder: Path):
    participant_id = extract_participant_id_from_folder(participant_folder.name)
    recording_suffix = extract_recording_suffix_from_folder(participant_folder.name)

    if participant_id is None:
        return [], [{"participant_folder": participant_folder.name, "recording_suffix": recording_suffix, "status": "invalid_participant_id"}]

    q_rows = questionnaires_df[questionnaires_df["ID"] == participant_id]
    d_rows = details_df[details_df["ID"] == participant_id]

    if q_rows.empty:
        return [], [{"participant_id": participant_id, "participant_folder": participant_folder.name, "recording_suffix": recording_suffix, "status": "missing_questionnaire"}]

    if d_rows.empty:
        return [], [{"participant_id": participant_id, "participant_folder": participant_folder.name, "recording_suffix": recording_suffix, "status": "missing_details"}]

    q_row = q_rows.iloc[0]
    d_row = d_rows.iloc[0]

    video_start, video_end, timing_source = estimate_video_start_end_timestamps(participant_folder, d_row)

    if pd.isna(video_start) or pd.isna(video_end):
        return [], [{
            "participant_id": participant_id,
            "participant_folder": participant_folder.name,
            "recording_suffix": recording_suffix,
            "status": "missing_video_start_or_end",
            "timing_source": timing_source,
        }]

    schedule_df = build_split6_schedule(video_start, video_end)
    if schedule_df.empty:
        return [], [{
            "participant_id": participant_id,
            "participant_folder": participant_folder.name,
            "recording_suffix": recording_suffix,
            "status": "invalid_schedule",
            "timing_source": timing_source,
        }]

    selected_files, file_selection_logs = select_participant_files(participant_folder)
    participant_rows = []
    logs = []

    for fs_log in file_selection_logs:
        fs_log.update({
            "participant_id": participant_id,
            "participant_folder": participant_folder.name,
            "recording_suffix": recording_suffix,
            "scene_name": "file_selection",
            "timing_source": timing_source,
        })
        logs.append(fs_log)

    for scene_name in EMPATHY_SCENES:
        label_idx = SCENE_TO_LABEL_INDEX[scene_name]
        target_valence = q_row.get(f"valence{label_idx}", np.nan)
        target_arousal = q_row.get(f"arousal{label_idx}", np.nan)
        target_discomfort = q_row.get(f"discomfort{label_idx}", np.nan)

        if pd.isna(target_valence) or pd.isna(target_arousal):
            logs.append({
                "participant_id": participant_id,
                "participant_folder": participant_folder.name,
                "recording_suffix": recording_suffix,
                "scene_name": scene_name,
                "status": "missing_target_label",
                "timing_source": timing_source,
            })
            continue

        scene_info = schedule_df[schedule_df["scene_name"] == scene_name].iloc[0]
        window_start = float(scene_info["window_start"])
        window_end = float(scene_info["window_end"])

        row = {
            "participant_id": participant_id,
            "participant_folder": participant_folder.name,
            "recording_suffix": recording_suffix,
            "video_set": q_row.get("video", np.nan),
            "scene_name": scene_name,
            "scene_index": label_idx,
            "target_valence": target_valence,
            "target_arousal": target_arousal,
            "target_discomfort": target_discomfort,
            "window_start": window_start,
            "window_end": window_end,
            "window_duration_seconds": window_end - window_start,
            "segment_duration_seconds": float(scene_info["segment_duration_seconds"]),
            "trim_seconds_used": float(scene_info["trim_seconds_used"]),
            "timing_source": timing_source,
            "feature_extraction_strategy": "split6_trimmed_fallback",
        }

        for file_path, file_type in selected_files:
            try:
                features, status = extract_features_from_file(file_path, file_type, window_start, window_end)
                row.update(features)
            except Exception as e:
                status = {"file": file_path.name, "file_type": file_type, "status": "error", "error": str(e), "rows_in_window": 0}

            status.update({
                "participant_id": participant_id,
                "participant_folder": participant_folder.name,
                "recording_suffix": recording_suffix,
                "scene_name": scene_name,
                "timing_source": timing_source,
            })
            logs.append(status)

        participant_rows.append(row)

    return participant_rows, logs


def run_feature_extraction():
    folders_to_process = participant_folders
    if MAX_PARTICIPANTS is not None:
        folders_to_process = folders_to_process[:MAX_PARTICIPANTS]

    all_rows = []
    all_logs = []

    print(f"Processing {len(folders_to_process)} participant folders...")

    for i, folder in enumerate(folders_to_process, start=1):
        print(f"[{i}/{len(folders_to_process)}] {folder.name}")
        rows, logs = process_one_participant(folder)
        all_rows.extend(rows)
        all_logs.extend(logs)
        gc.collect()

    features_df = pd.DataFrame(all_rows)
    logs_df = pd.DataFrame(all_logs)
    return features_df, logs_df


## feature extraction

In [16]:
features_df, logs_df = run_feature_extraction()

print("Feature dataset shape:", features_df.shape)
display(features_df.head())

print("Log status counts:")
if len(logs_df) > 0 and "status" in logs_df.columns:
    display(logs_df["status"].value_counts(dropna=False).to_frame("count"))
else:
    print("No logs available.")


Processing 106 participant folders...
[1/106] participant1
[2/106] participant2
[3/106] participant3
[4/106] participant4
[5/106] participant5
[6/106] participant6
[7/106] participant7
[8/106] participant8
[9/106] participant9
[10/106] participant10
[11/106] participant11
[12/106] participant12-1
[13/106] participant12-2
[14/106] participant13
[15/106] participant14
[16/106] participant15
[17/106] participant16
[18/106] participant17
[19/106] participant18
[20/106] participant19
[21/106] participant20
[22/106] participant21
[23/106] participant22
[24/106] participant23
[25/106] participant24
[26/106] participant25
[27/106] participant26
[28/106] participant27
[29/106] participant28
[30/106] participant29
[31/106] participant30
[32/106] participant31
[33/106] participant32
[34/106] participant33
[35/106] participant34
[36/106] participant35
[37/106] participant36
[38/106] participant37
[39/106] participant38
[40/106] participant39
[41/106] participant40
[42/106] participant41
[43/106] p

,participant_id,participant_folder,recording_suffix,video_set,scene_name,scene_index,target_valence,target_arousal,target_discomfort,window_start,window_end,window_duration_seconds,segment_duration_seconds,trim_seconds_used,timing_source,feature_extraction_strategy,breathingrate_breathingrate_min,breathingrate_breathingrate_max,breathingrate_imu_motionintensity_min,breathingrate_imu_motionintensity_max,qc_breathingrate_ppg_quality_min,qc_breathingrate_ppg_quality_mean,emgactivation_emg_amplitude_zygo_weighted_min,emgactivation_emg_amplitude_zygo_weighted_max,emgactivation_emg_amplitude_orbi_weighted_min,emgactivation_emg_amplitude_orbi_weighted_max,emgactivation_emg_amplitude_front_weighted_min,emgactivation_emg_amplitude_front_weighted_max,emgactivation_emg_amplitude_corr_weighted_min,emgactivation_emg_amplitude_corr_weighted_max,expression_expression_intensity_min,expression_expression_intensity_max,expression_neutral_intensity_min,expression_neutral_intensity_max,facialactivation_facialactivation_min,facialactivation_facialactivation_max,hrv_hrv_mean_hr_min,hrv_hrv_mean_hr_max,hrv_hrv_rr_min,hrv_hrv_rr_max,hrv_hrv_sdnn_min,hrv_hrv_sdnn_max,hrv_hrv_sdsd_min,hrv_hrv_sdsd_max,hrv_hrv_rmssd_min,hrv_hrv_rmssd_max,hrv_imu_motionintensity_min,hrv_imu_motionintensity_max,qc_hrv_ppg_quality_min,qc_hrv_ppg_quality_mean,expression_smile_intensity_min,expression_smile_intensity_max,expression_eyebrow_raise_intensity_min,expression_eyebrow_raise_intensity_max,expression_frown_intensity_min,expression_frown_intensity_max
0,1,participant1,0,3,empathy_scene_1,1,4,3,3,1.703164e+09,1.703165e+09,230.0,310.0,40.0,details_start_end,split6_trimmed_fallback,10.461313,15.128110,0.0,0.153846,0.000000,0.977699,-0.066996,24.218302,1.578087,92.229076,2.812798,52.798756,0.413829,54.800423,0.0,0.0,0.0,0.0,0.030240,0.290159,91.101695,96.443228,622.127660,658.604651,45.445454,214.876933,68.912592,270.928901,69.012975,270.965352,0.0,0.133333,0.632841,0.970686,NaN,NaN,NaN,NaN,NaN,NaN
1,1,participant1,0,3,empathy_scene_2,2,3,2,3,1.703165e+09,1.703165e+09,230.0,310.0,40.0,details_start_end,split6_trimmed_fallback,10.616741,15.623618,0.0,0.230769,0.997654,0.999898,-0.102239,28.426819,1.441766,93.195526,0.935980,55.004483,0.043085,54.107571,0.0,0.0,0.0,0.0,0.019762,0.396651,87.110482,100.884956,594.736842,688.780488,34.871192,109.335817,47.809144,150.926391,47.809144,151.034528,0.0,0.200000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN
2,1,participant1,0,3,empathy_scene_3,3,0,2,0,1.703165e+09,1.703165e+09,230.0,310.0,40.0,details_start_end,split6_trimmed_fallback,9.446169,14.115516,0.0,0.153846,0.000000,0.875416,1.569098,176.287715,2.820233,125.440448,-0.464864,21.345565,-1.266525,16.900548,0.0,86.7,0.0,0.0,0.025834,0.719932,86.124402,110.526316,542.857143,696.666667,49.156144,207.506047,62.444420,296.878531,62.449980,296.940925,0.0,0.133333,0.674555,0.940022,5.6,86.7,NaN,NaN,NaN,NaN
3,1,participant1,0,3,empathy_scene_4,4,4,3,3,1.703165e+09,1.703165e+09,230.0,310.0,40.0,details_start_end,split6_trimmed_fallback,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,participant2,0,1,empathy_scene_1,1,3,3,3,1.704457e+09,1.704457e+09,200.0,280.0,40.0,details_start_end,split6_trimmed_fallback,9.931256,15.976462,0.0,0.153846,1.000000,1.000000,-0.924488,43.749788,-0.533047,70.130081,0.067359,16.575341,1.361073,12.265237,0.0,21.9,0.0,0.0,0.011568,0.269307,80.394922,86.677368,692.222222,746.315789,42.288263,105.070848,63.407514,150.074056,63.407514,150.111070,0.0,0.133333,1.000000,1.000000,1.2,21.9,NaN,NaN,NaN,NaN


Log status counts:


,count
status,
ok,2090
excluded_affect_output,316
unknown_file_type,84
optional_no_time_column,77
empty_window,30


## Sanity checks

This section checks whether the extracted dataset is consistent.

The most important checks are:

- four rows per valid participant;
- equal number of rows for empathy scenes 1–4;
- no missing valence/arousal targets;
- reasonable window duration;
- missing feature ratios.


In [18]:
if len(features_df) > 0:
    print("Rows by scene:")
    display(features_df["scene_name"].value_counts().sort_index())

    print("\nRows by participant:")
    rows_by_participant = features_df.groupby(["participant_id", "participant_folder"]).size().to_frame("n_rows")
    display(rows_by_participant)

    print("\nMissing targets:")
    display(features_df[["target_valence", "target_arousal"]].isna().sum())

    print("\nWindow duration summary:")
    display(features_df["window_duration_seconds"].describe())

    print("\nTiming source counts:")
    display(features_df["timing_source"].value_counts(dropna=False).to_frame("count"))

    metadata_cols = {
        "participant_id", "participant_folder", "recording_suffix", "video_set",
        "scene_name", "scene_index", "target_valence", "target_arousal", "target_discomfort",
        "window_start", "window_end", "window_duration_seconds",
        "segment_duration_seconds", "trim_seconds_used", "timing_source",
        "feature_extraction_strategy",
    }

    feature_cols = [c for c in features_df.columns if c not in metadata_cols and not c.startswith("qc_")]

    print("\nNumber of feature columns:", len(feature_cols))
    print("First feature columns:", feature_cols[:30])
    print("Segment duration summary before trimming:")
    display(features_df["segment_duration_seconds"].describe())

    print("Window duration summary after trimming:")
    display(features_df["window_duration_seconds"].describe())

    print("Average duration by scene:")
    display(
        features_df
        .groupby("scene_name")[["segment_duration_seconds", "window_duration_seconds"]]
        .describe()
    )

    if feature_cols:
        missing_ratios = features_df[feature_cols].isna().mean().sort_values(ascending=False)
        display(missing_ratios.head(40).to_frame("missing_ratio"))
else:
    print("No features extracted.")

    


Rows by scene:


scene_name
empathy_scene_1    106
empathy_scene_2    106
empathy_scene_3    106
empathy_scene_4    106
Name: count, dtype: int64


Rows by participant:


,,n_rows
participant_id,participant_folder,
1,participant1,4
2,participant2,4
3,participant3,4
4,participant4,4
5,participant5,4
...,...,...
101,participant101,4
102,participant102,4
103,participant103,4



Missing targets:


...

Window duration summary after trimming:


count      424.000000
mean       203.908288
std       1039.760002
min         30.000000
25%         90.000000
50%        100.000000
75%        110.000000
max      10842.666505
Name: window_duration_seconds, dtype: float64

Average duration by scene:


segment_duration_seconds                                                                        window_duration_seconds                                                     \
                                   count        mean          std        min    25%    50%    75%           max                   count        mean          std   min   25%    50%    75%   
scene_name                                                                                                                                                                                   
empathy_scene_1                    106.0  283.273618  1043.593661  43.379172  170.0  180.0  190.0  10922.666505                   106.0  203.908288  1043.466823  30.0  90.0  100.0  110.0   
empathy_scene_2                    106.0  283.273618  1043.593661  43.379172  170.0  180.0  190.0  10922.666505                   106.0  203.908288  1043.466823  30.0  90.0  100.0  110.0   
empathy_scene_3                    106.0  283.273618  1043.593661  43.379172  170.0  180.0  190.0  10922.666505                   106.0  203.908288  1043.466823  30.0  90.0  100.0  110.0   
empathy_scene_4                    106.0  283.273618  1043.593661  43.379172  170.0  180.0  190.0  10922.666505                   106.0  203.908288  1043.466823  30.0  90.0  100.0  110.0   

                               
                          max  
scene_name                     
empathy_scene_1  10842.666505  
empathy_scene_2  10842.666505  
empathy_scene_3  10842.666505  
empathy_scene_4  10842.666505

,missing_ratio
expression_frown_intensity_max,0.875000
expression_frown_intensity_min,0.875000
expression_eyebrow_raise_intensity_max,0.834906
expression_eyebrow_raise_intensity_min,0.834906
expression_smile_intensity_max,0.580189
expression_smile_intensity_min,0.580189
breathingrate_breathingrate_max,0.018868
breathingrate_breathingrate_min,0.018868
emgactivation_emg_amplitude_front_weighted_min,0.014151
emgactivation_emg_amplitude_front_weighted_max,0.014151


In [19]:
duration_check = features_df[
    [
        "participant_id",
        "participant_folder",
        "scene_name",
        "segment_duration_seconds",
        "trim_seconds_used",
        "window_duration_seconds"
    ]
].copy()

display(duration_check.head(40))

,participant_id,participant_folder,scene_name,segment_duration_seconds,trim_seconds_used,window_duration_seconds
0,1,participant1,empathy_scene_1,310.000000,40.0,230.000000
1,1,participant1,empathy_scene_2,310.000000,40.0,230.000000
2,1,participant1,empathy_scene_3,310.000000,40.0,230.000000
3,1,participant1,empathy_scene_4,310.000000,40.0,230.000000
4,2,participant2,empathy_scene_1,280.000000,40.0,200.000000
5,2,participant2,empathy_scene_2,280.000000,40.0,200.000000
6,2,participant2,empathy_scene_3,280.000000,40.0,200.000000
7,2,participant2,empathy_scene_4,280.000000,40.0,200.000000
8,3,participant3,empathy_scene_1,200.000000,40.0,120.000000
9,3,participant3,empathy_scene_2,200.000000,40.0,120.000000


## Check missing participants

This section checks which questionnaire participants were not included in the extracted dataset.

After adding the fallback based on the last sensor timestamp, fewer participants should be missing. If participants are still missing, the logs should explain whether the problem is missing questionnaire data, missing details, missing sensor timestamps, or invalid schedule construction.


In [20]:
if len(features_df) > 0:
    processed_ids = set(features_df["participant_id"].dropna().astype(int).unique())
    all_questionnaire_ids = set(questionnaires_df["ID"].dropna().astype(int).unique())
    missing_ids = sorted(all_questionnaire_ids - processed_ids)

    print("Number of questionnaire participants:", len(all_questionnaire_ids))
    print("Number of processed participants:", len(processed_ids))
    print("Missing participant IDs:", missing_ids)

    if len(missing_ids) > 0 and len(logs_df) > 0:
        display(logs_df[logs_df["participant_id"].isin(missing_ids)].head(100))
else:
    print("No extracted features available.")


Number of questionnaire participants: 105
Number of processed participants: 105
Missing participant IDs: []


## Quality control: remove unreliable timing windows

In this section, windows with unreliable timing are removed. Participants without a valid end video time can produce unrealistic window durations when the last available sensor timestamp is used as fallback (after seeing the sanity check outputs - this is a must). Since the dataset still contains enough valid participants, only rows with reliable start and end timing from `final_data_details.csv` are kept for the main analysis.

In [21]:
print("Shape before timing QC:", features_df.shape)

print("Timing source counts before QC:")
display(features_df["timing_source"].value_counts(dropna=False).to_frame("count"))

print("Window duration summary before QC:")
display(features_df["window_duration_seconds"].describe())

# Keep only rows where both video start and video end came from final_data_details.csv
features_df = features_df[features_df["timing_source"] == "details_start_end"].copy()

print("Shape after keeping only reliable timing rows:", features_df.shape)

print("Timing source counts after QC:")
display(features_df["timing_source"].value_counts(dropna=False).to_frame("count"))

print("Window duration summary after QC:")
display(features_df["window_duration_seconds"].describe())

Shape before timing QC: (424, 56)
Timing source counts before QC:


,count
timing_source,
details_start_end,392
fallback_last_sensor_time,32


Window duration summary before QC:


count      424.000000
mean       203.908288
std       1039.760002
min         30.000000
25%         90.000000
50%        100.000000
75%        110.000000
max      10842.666505
Name: window_duration_seconds, dtype: float64

Shape after keeping only reliable timing rows: (392, 56)
Timing source counts after QC:


,count
timing_source,
details_start_end,392


Window duration summary after QC:


count    392.000000
mean     103.673469
std       22.992838
min       80.000000
25%       90.000000
50%      100.000000
75%      110.000000
max      230.000000
Name: window_duration_seconds, dtype: float64

# Remove participants with missing sensor data 

Specifically participant 1 and 2 

In [26]:
#remove participants with incomplete sensor data

metadata_cols = {
    "participant_id", "participant_folder", "recording_suffix", "video_set",
    "scene_name", "scene_index", "target_valence", "target_arousal", "target_discomfort",
    "window_start", "window_end", "window_duration_seconds",
    "segment_duration_seconds", "trim_seconds_used", "timing_source",
    "feature_extraction_strategy",
}

expression_type_cols = [
    c for c in features_df.columns
    if c.startswith("expression_")
    and c not in ["expression_expression_intensity_min", "expression_expression_intensity_max"]
]

core_feature_cols = [
    c for c in features_df.columns
    if c not in metadata_cols
    and not c.startswith("qc_")
    and c not in expression_type_cols
]

features_df["n_core_available_features"] = features_df[core_feature_cols].notna().sum(axis=1)

# Find rows where no real sensor features were extracted
empty_sensor_rows = features_df[features_df["n_core_available_features"] == 0]

print("Rows with no core sensor features:")
display(
    empty_sensor_rows[
        ["participant_id", "participant_folder", "scene_name", "window_duration_seconds"]
    ]
)

# Drop the full participant if any of their scenes has no sensor features
participants_to_drop = sorted(empty_sensor_rows["participant_id"].unique())

print("Participants to drop completely:", participants_to_drop)

features_df = features_df[~features_df["participant_id"].isin(participants_to_drop)].copy()

print("Shape after dropping incomplete participants:", features_df.shape)

print("Rows by scene after dropping incomplete participants:")
display(features_df["scene_name"].value_counts().sort_index())

print("Rows by participant after dropping incomplete participants:")
display(
    features_df
    .groupby(["participant_id", "participant_folder"])
    .size()
    .to_frame("n_rows")
)

Rows with no core sensor features:


,participant_id,participant_folder,scene_name,window_duration_seconds
3,1,participant1,empathy_scene_4,230.0
7,2,participant2,empathy_scene_4,200.0


Participants to drop completely: [np.int64(1), np.int64(2)]
Shape after dropping incomplete participants: (384, 57)
Rows by scene after dropping incomplete participants:


scene_name
empathy_scene_1    96
empathy_scene_2    96
empathy_scene_3    96
empathy_scene_4    96
Name: count, dtype: int64

Rows by participant after dropping incomplete participants:


,,n_rows
participant_id,participant_folder,
3,participant3,4
4,participant4,4
5,participant5,4
13,participant13,4
14,participant14,4
...,...,...
101,participant101,4
102,participant102,4
103,participant103,4


In [22]:
print("Rows by scene after QC:")
display(features_df["scene_name"].value_counts().sort_index())

print("Rows by participant after QC:")
display(
    features_df
    .groupby(["participant_id", "participant_folder"])
    .size()
    .to_frame("n_rows")
)

processed_ids = set(features_df["participant_id"].dropna().astype(int).unique())
all_questionnaire_ids = set(questionnaires_df["ID"].dropna().astype(int).unique())

missing_ids = sorted(all_questionnaire_ids - processed_ids)

print("Number of processed participants after QC:", len(processed_ids))
print("Missing participant IDs after QC:", missing_ids)

Rows by scene after QC:


scene_name
empathy_scene_1    98
empathy_scene_2    98
empathy_scene_3    98
empathy_scene_4    98
Name: count, dtype: int64

Rows by participant after QC:


,,n_rows
participant_id,participant_folder,
1,participant1,4
2,participant2,4
3,participant3,4
4,participant4,4
5,participant5,4
...,...,...
101,participant101,4
102,participant102,4
103,participant103,4


Number of processed participants after QC: 98
Missing participant IDs after QC: [np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


## expression-specific missing values

This section fills missing expression specific intensity features with 0.

Reasoning:

- if `expression_smile_intensity_max` is missing, it usually means that smile was not detected in that scene window;
- therefore, the expression-specific intensity can be treated as 0 for modelling.

This is applied only to expression specific columns.


In [23]:
if len(features_df) > 0:
    expression_cols = [c for c in features_df.columns if c.startswith("expression_")]
    global_expression_cols = {
        "expression_expression_intensity_min",
        "expression_expression_intensity_max",
    }
    expression_type_cols = [c for c in expression_cols if c not in global_expression_cols]

    features_df[expression_type_cols] = features_df[expression_type_cols].fillna(0)

    print("Expression-specific missing values filled with 0.")
    print("Number of expression-specific columns:", len(expression_type_cols))

    if expression_type_cols:
        display(features_df[expression_type_cols].isna().mean().sort_values(ascending=False).head(20).to_frame("missing_ratio_after_fill"))
else:
    print("No features available.")


Expression-specific missing values filled with 0.
Number of expression-specific columns: 8


,missing_ratio_after_fill
expression_neutral_intensity_min,0.0
expression_neutral_intensity_max,0.0
expression_smile_intensity_min,0.0
expression_smile_intensity_max,0.0
expression_eyebrow_raise_intensity_min,0.0
expression_eyebrow_raise_intensity_max,0.0
expression_frown_intensity_min,0.0
expression_frown_intensity_max,0.0


## duplicate participant check

This section checks whether any participant ID appears more than four times.

Usually, each participant should have exactly four rows, one for each empathy scene. If a participant has multiple folders, for example `participant12-1` and `participant12-2`, the dataset may contain more than four rows for that participant. In that case, inspect the rows and decide whether one recording should be kept.


In [24]:
if len(features_df) > 0:
    counts_per_id = features_df.groupby("participant_id").size().sort_values(ascending=False)
    duplicate_or_extra = counts_per_id[counts_per_id > 4]

    if len(duplicate_or_extra) > 0:
        print("Participants with more than 4 rows:")
        display(duplicate_or_extra.to_frame("n_rows"))
        display(features_df[features_df["participant_id"].isin(duplicate_or_extra.index)].sort_values(["participant_id", "participant_folder", "scene_index"]))
    else:
        print("All processed participants have at most 4 rows.")
else:
    print("No features available.")


All processed participants have at most 4 rows.


## Save outputs

This section saves:

1. the final min/max feature dataset;
2. the feature extraction log file.

The saved feature dataset will be used by the next notebooks: EDA, statistical analysis, and machine learning for valence and arousal prediction.


In [27]:
if len(features_df) > 0:
    features_df.to_csv(FINAL_OUTPUT_PATH, index=False)
    logs_df.to_csv(LOG_OUTPUT_PATH, index=False)

    print("Saved feature dataset:", FINAL_OUTPUT_PATH)
    print("Saved logs:", LOG_OUTPUT_PATH)
    print("Final dataset shape:", features_df.shape)
else:
    print("No features to save.")


Saved feature dataset: /Users/victoriaprojkova/EmteqPRO-VR/outputs/video_level_valence_arousal_minmax.csv
Saved logs: /Users/victoriaprojkova/EmteqPRO-VR/outputs/feature_extraction_logs.csv
Final dataset shape: (384, 57)
